In [ ]:
import requests

In [ ]:
URL_DOCS = "https://climate.ec.europa.eu/eu-action/european-climate-law_en#documentation"
response = requests.get(URL_DOCS,headers={"User-Agent": "Mozilla/5.0"})


In [ ]:
response.raise_for_status()  # Check if the request was successful

In [ ]:
response.text

In [ ]:

import asyncio
import re
from urllib.parse import urlparse
from playwright.async_api import async_playwright


async def get_doc_links_by_section(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="networkidle")
        await page.wait_for_timeout(3000)

        # Click only the accordion toggles inside the Documentation section
        toggles = await page.query_selector_all('#documentation ~ * .ecl-accordion__toggle')
        for toggle in toggles:
            try:
                await toggle.click()
                await page.wait_for_timeout(500)
            except:
                pass

        # Extract links grouped by accordion item title, excluding generic "Download" links
        sections = await page.evaluate("""() => {
    const docHeading = document.querySelector('#documentation');
    if (!docHeading) return {};

    let container = docHeading.parentElement;

    // Look downward and sideways, but do not climb too far
    while (container && !container.querySelector('.ecl-accordion__item')) {
        container = container.nextElementSibling || container.parentElement;
    }

    if (!container) return {};

    const result = {};

    container.querySelectorAll('.ecl-accordion__item').forEach(item => {
        const titleEl = item.querySelector('.ecl-accordion__toggle');
        const title = titleEl ? titleEl.textContent.trim() : 'Unknown';

        const links = [...item.querySelectorAll('a[href]')]
            .map(a => ({
                href: a.href,
                text: a.textContent.trim()
            }))
            .filter(l =>
                l.text &&
                !['download', 'preview'].includes(l.text.toLowerCase())
            );

        if (links.length) result[title] = links;
    });

    return result;
}""")

        await browser.close()
        return sections

doc_sections = await get_doc_links_by_section(URL_DOCS)

# Normalise EUR-Lex links inline
for section, links in doc_sections.items():
    print(f"\n=== {section} ===")
    for l in links:
        print(f"  {l['text']}  -->  {l['href']}")
        if "eur-lex" in l['href']:
            if ("HTML" not in l['href']) and ("/TXT/" in l['href']):
                l['href'] = l['href'].replace("/TXT/", "/TXT/HTML/")
                print(f"    Updated link: {l['href']}")


class UrlNormalizer:
    def normalize(self, url: str) -> str:
        if "eur-lex" in url and "/TXT/" in url and "/TXT/HTML/" not in url:
            return url.replace("/TXT/", "/TXT/HTML/")
        return url


class MetadataEnricher:
    def __init__(self, url_normalizer: UrlNormalizer):
        self.url_normalizer = url_normalizer

    def enrich(self, sections):
        enriched = {}
        for section, links in sections.items():
            enriched[section] = [self._enrich_link(link) for link in links]
        return enriched

    def _enrich_link(self, link):
        title = link["text"]
        url = link["href"]
        return {
            "title": title,
            "url": url,
            "type": self._detect_type(title),
            "year": self._extract_year(title, url),
            "identifier": self._extract_identifier(url),
            "source": self._extract_source(url),
            "format": self._detect_format(url),
            "topic": self._detect_topic(title),
        }

    def _detect_type(self, title):
        t = title.lower()
        if "regulation" in t: return "regulation"
        if "proposal" in t: return "proposal"
        if "communication" in t: return "communication"
        if "staff working document" in t: return "staff_document"
        if "factsheet" in t: return "factsheet"
        if "press release" in t: return "press_release"
        if "questions" in t or "q&a" in t: return "qa"
        return "other"

    def _extract_year(self, title, url):
        match = re.search(r"\b(20\d{2})\b", title + url)
        print(f"Extracting year from: {title} | {url} --> Found: {match.group(1) if match else 'None'}")
        return int(match.group(1)) if match else None

    def _extract_identifier(self, url):
        if "CELEX:" in url:
            return url.split("CELEX:")[-1]
        if "COM:" in url:
            return url.split("COM:")[-1]
        match = re.search(r"(SWD_\d+_\d+)", url)
        return match.group(1) if match else None

    def _extract_source(self, url):
        domain = urlparse(url).netloc
        if "eur-lex" in domain: return "eur-lex"
        if "ec.europa.eu" in domain: return "commission"
        if "climate.ec.europa.eu" in domain: return "climate-portal"
        return "other"

    def _detect_format(self, url):
        if ".pdf" in url: return "pdf"
        if "presscorner" in url: return "press_page"
        return "html"

    def _detect_topic(self, title):
        t = title.lower()
        if "climate law" in t: return "climate_law"
        if "2040" in t: return "climate_target_2040"
        if "industrial deal" in t: return "clean_industrial_deal"
        return "general"


enricher = MetadataEnricher(url_normalizer=UrlNormalizer())
enriched_sections = enricher.enrich(doc_sections)


In [ ]:
from openai import OpenAI

openai_client = OpenAI()

In [ ]:
import io
import json
import os
import tempfile
import uuid
from markitdown import MarkItDown

#  In-memory content cache (avoids passing large HTML/Markdown through the LLM) 
_content_cache: dict = {}

#  Downloadable document extensions 
_DOC_EXTENSIONS = (".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx", ".odt", ".rtf")

# Keywords that indicate a button triggers a file download
_DOWNLOAD_BUTTON_KEYWORDS = ("download", "télécharger", "descargar", "scarica", "herunterladen", "pobierz")


def _is_pdf_url(url: str) -> bool:
    """Heuristic: treat as PDF if the URL path ends with .pdf or contains a known download pattern."""
    lower = url.lower().split("?")[0]
    return lower.endswith(".pdf") or "/pdf" in lower or ("download" in lower and ".pdf" in url.lower())


def _is_document_url(href: str) -> bool:
    """Return True if the href looks like a downloadable document.
    Signals:
      - URL path ends with a known document extension (.pdf, .docx, etc.)
      - URL contains '/download' or 'download=' (common download-endpoint patterns)
      - URL path contains both 'document' and 'download' path segments
        (EU Commission pattern: /document/download/<uuid>?filename=...pdf)
      - Passes the _is_pdf_url heuristic
    """
    lower = href.lower()
    path = lower.split("?")[0]
    return (
        any(path.endswith(ext) for ext in _DOC_EXTENSIONS)
        or "/download" in lower
        or "download=" in lower
        or ("document" in path and "download" in path)  # EU Commission: /document/download/<uuid>
        or _is_pdf_url(href)
    )


def _is_download_button(text: str) -> bool:
    """Return True if button text suggests it triggers a file download."""
    return any(kw in text.lower() for kw in _DOWNLOAD_BUTTON_KEYWORDS)


def _detect_format_from_url(url: str) -> str:
    """Guess the binary format from a URL's filename/extension."""
    path = url.lower().split("?")[0]
    if path.endswith(".pdf") or "pdf" in path:
        return "pdf"
    return "binary"



In [ ]:

async def playwright_get_page_snapshot(url: str) -> dict:
    """Open a page and return its title, all visible links, buttons, download_buttons,
    and whether the page appears to contain downloadable documents or download buttons.
    If the URL is a direct file download (PDF or EU Commission /document/download/<uuid>),
    skip browser navigation and return immediately."""
    if _is_document_url(url):
        filename = url.rstrip("/").split("/")[-1].split("?")[0] or "document"
        # Try to get a better filename from query string ?filename=...
        if "filename=" in url.lower():
            import urllib.parse
            qs = urllib.parse.parse_qs(urllib.parse.urlparse(url).query)
            filename = qs.get("filename", [filename])[0]
        return {
            "title": filename,
            "url": url,
            "links": [{"index": 0, "text": filename, "href": url}],
            "buttons": [],
            "download_buttons": [],
            "has_downloadable_documents": True,
            "has_download_buttons": False,
            "document_links": [{"text": filename, "href": url}],
            "note": "Direct download URL; use click_and_capture with this href to fetch it."
        }

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="networkidle")
        await page.wait_for_timeout(2000)

        snapshot = await page.evaluate("""() => {
            // Regular <a href> links (up to 60)
            const links = [...document.querySelectorAll('a[href]')]
                .map((a, i) => ({
                    index: i,
                    text: a.textContent.trim().slice(0, 120),
                    href: a.href
                }))
                .filter(l => l.text)
                .slice(0, 60);

            // <a href role="button"> elements with 'document' AND 'download' in href.
            // EU Commission download pattern:
            //   <a href="/document/download/<uuid>?filename=...pdf" role="button">
            // Collected separately so they are never cut off by the 60-link limit.
            const anchorButtonLinks = [...document.querySelectorAll('a[href][role="button"]')]
                .map(a => ({
                    text: (a.textContent.trim().slice(0, 120)
                           || a.getAttribute('aria-label')
                           || 'Download'),
                    href: a.href
                }))
                .filter(l => {
                    const h = l.href.toLowerCase();
                    return h.includes('document') && h.includes('download');
                })
                .slice(0, 20);

            // Real <button> / <input> elements (NOT <a role="button">)
            const buttons = [...document.querySelectorAll(
                'button, input[type="button"], input[type="submit"]'
            )]
                .map((b, i) => ({
                    index: i,
                    text: (b.textContent || b.value || '').trim().slice(0, 120)
                }))
                .filter(b => b.text)
                .slice(0, 20);

            return {
                title: document.title,
                url: window.location.href,
                links: links,
                anchor_button_links: anchorButtonLinks,
                buttons: buttons
            };
        }""")

        await browser.close()

    # Identify download links from regular <a href> links
    doc_links = [
        {"text": l["text"], "href": l["href"]}
        for l in snapshot.get("links", [])
        if _is_document_url(l["href"])
    ]

    # Merge in <a role="button"> download links, deduplicated by href
    existing_hrefs = {d["href"] for d in doc_links}
    for al in snapshot.get("anchor_button_links", []):
        if al["href"] not in existing_hrefs:
            doc_links.append({"text": al["text"], "href": al["href"]})
            existing_hrefs.add(al["href"])

    # Identify download buttons (real <button> elements whose text says "download")
    download_buttons = [
        b for b in snapshot.get("buttons", [])
        if _is_download_button(b["text"])
    ]

    snapshot["document_links"] = doc_links
    snapshot["download_buttons"] = download_buttons
    snapshot["has_downloadable_documents"] = len(doc_links) > 0
    snapshot["has_download_buttons"] = len(download_buttons) > 0

    return snapshot


In [ ]:
async def playwright_click_download_button(url: str, button_text: str) -> dict:
    """
    Navigate to url, click the button whose text matches button_text, and intercept
    the resulting file download using Playwright's download API.
    Falls back to capturing the page HTML if no download is triggered.
    Returns a content_id for use with convert_to_markdown.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()
        page = await context.new_page()
        await page.goto(url, wait_until="networkidle")
        await page.wait_for_timeout(1500)

        selector = (
            f'button:has-text("{button_text}"), '
            f'[role="button"]:has-text("{button_text}"), '
            f'input[value="{button_text}"]'
        )
        button = await page.query_selector(selector)
        if button is None:
            await browser.close()
            return {"error": f"No button found matching text '{button_text}'"}

        try:
            async with page.expect_download(timeout=15000) as download_info:
                await button.click()
            download = await download_info.value

            suffix = os.path.splitext(download.suggested_filename)[1] or ".bin"
            tmp = tempfile.NamedTemporaryFile(suffix=suffix, delete=False)
            await download.save_as(tmp.name)
            tmp.close()
            await browser.close()

            title = download.suggested_filename
            fmt = "pdf" if suffix.lower() == ".pdf" else "binary"
            content_id = uuid.uuid4().hex[:8]
            _content_cache[content_id] = {"format": fmt, "tmp_path": tmp.name, "title": title}
            return {"url": url, "title": title, "format": fmt, "filename": title, "content_id": content_id}

        except Exception:
            await page.wait_for_load_state("networkidle")
            await page.wait_for_timeout(2000)
            final_url = page.url
            title = await page.title()
            html = await page.content()
            await browser.close()

            content_id = uuid.uuid4().hex[:8]
            _content_cache[content_id] = {"format": "html", "html": html, "title": title}
            return {"url": final_url, "title": title, "format": "html", "content_id": content_id}


In [ ]:

async def playwright_click_and_capture(url: str, link_href: str) -> dict:
    """
    Navigate to url, click the link, fetch the resulting page via Playwright,
    store the raw content in the cache, and return a content_id.
    For HTML pages, stores the full page HTML.
    For document download URLs (PDFs, EU Commission /document/download/<uuid>, etc.),
    uses context.request.get() directly instead of browser navigation.
    """
    is_download = _is_document_url(link_href)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context()

        if is_download:
            response = await context.request.get(link_href)
            file_bytes = await response.body()
            # Try to get a filename from Content-Disposition header or URL
            content_disposition = response.headers.get("content-disposition", "")
            filename = None
            if "filename=" in content_disposition:
                filename = content_disposition.split("filename=")[-1].strip().strip('"')
            if not filename:
                filename = link_href.rstrip("/").split("/")[-1].split("?")[0] or "document"
                if "filename=" in link_href.lower():
                    import urllib.parse
                    qs = urllib.parse.parse_qs(urllib.parse.urlparse(link_href).query)
                    filename = qs.get("filename", [filename])[0]
            await browser.close()

            suffix = os.path.splitext(filename)[1].lower() or ".bin"
            tmp = tempfile.NamedTemporaryFile(suffix=suffix, delete=False)
            tmp.write(file_bytes)
            tmp.close()
            fmt = "pdf" if suffix == ".pdf" else "binary"
            content_id = uuid.uuid4().hex[:8]
            _content_cache[content_id] = {"format": fmt, "tmp_path": tmp.name, "title": filename}
            return {"url": link_href, "title": filename, "format": fmt, "content_id": content_id}

        page = await context.new_page()
        await page.goto(url, wait_until="networkidle")
        await page.wait_for_timeout(1500)

        try:
            await page.click(f'a[href="{link_href}"]')
            await page.wait_for_load_state("networkidle")
            await page.wait_for_timeout(2000)
        except Exception:
            await page.goto(link_href, wait_until="networkidle")
            await page.wait_for_timeout(2000)

        final_url = page.url
        title = await page.title()
        html = await page.content()
        await browser.close()

        content_id = uuid.uuid4().hex[:8]
        _content_cache[content_id] = {"format": "html", "html": html, "title": title}
        return {"url": final_url, "title": title, "format": "html", "content_id": content_id}


def convert_to_markdown(content_id: str) -> dict:
    """
    Convert cached Playwright-fetched content to Markdown using MarkItDown.
    For HTML: converts from the in-memory HTML string.
    For PDF/binary: converts from the temp file.
    Returns a new markdown_id for use with save_content_to_file.
    """
    cached = _content_cache.pop(content_id, None)
    if cached is None:
        return {"error": f"content_id '{content_id}' not found in cache"}

    md = MarkItDown()

    if cached["format"] == "html":
        result = md.convert_stream(
            io.BytesIO(cached["html"].encode("utf-8")),
            file_extension=".html"
        )
    else:
        result = md.convert(cached["tmp_path"])
        os.unlink(cached["tmp_path"])

    markdown_id = uuid.uuid4().hex[:8]
    _content_cache[markdown_id] = {"markdown": result.markdown, "title": cached["title"]}

    return {
        "markdown_id": markdown_id,
        "title": cached["title"],
        "length": len(result.markdown),
        "preview": result.markdown[:800]  # larger preview so the LLM can judge relevance
    }


In [ ]:


def save_content_to_file(markdown_id: str, filename: str, directory: str = "climate_policy_docs") -> dict:
    """Write cached Markdown content to a .md file."""
    cached = _content_cache.pop(markdown_id, None)
    if cached is None:
        return {"error": f"markdown_id '{markdown_id}' not found in cache"}

    os.makedirs(directory, exist_ok=True)
    filepath = os.path.join(directory, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(cached["markdown"])
    return {"saved": True, "path": filepath, "bytes": len(cached["markdown"].encode("utf-8"))}




In [ ]:

# Tool schemas

get_page_snapshot_tool = {
    "type": "function",
    "name": "get_page_snapshot",
    "description": (
        "Open a web page with a headless browser and return its title, "
        "all visible links (text + href), buttons, and four extra fields: "
        "'has_downloadable_documents' (bool), <a href> links whose href ends with a "
        "document extension (.pdf, .docx, etc.), contains '/download', 'download=', or "
        "matches the EU Commission pattern (/document/download/<uuid>); "
        "'document_links', those filtered links, including <a role='button'> anchors "
        "with 'document' and 'download' in the href; "
        "'has_download_buttons' (bool), <button> elements whose text contains 'download'; "
        "'download_buttons', those filtered buttons. "
        "Call this first on any page, and again after navigating, to decide whether the "
        "page has document links, download buttons, or is just an intermediate index."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url": {"type": "string", "description": "The URL of the page to inspect."}
        },
        "required": ["url"]
    }
}

click_and_capture_tool = {
    "type": "function",
    "name": "click_and_capture",
    "description": (
        "Navigate to a page, click a specific <a href> link by its href using Playwright, "
        "and store the fetched content (HTML or PDF) in a local cache. "
        "Use this for document_links returned by get_page_snapshot. "
        "For direct download URLs (e.g. EU Commission /document/download/<uuid>), "
        "the file is fetched via HTTP directly without browser navigation. "
        "Returns a content_id, pass it to convert_to_markdown next."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url":       {"type": "string", "description": "The starting page URL."},
            "link_href": {"type": "string", "description": "The exact href of the link to click."}
        },
        "required": ["url", "link_href"]
    }
}

click_download_button_tool = {
    "type": "function",
    "name": "click_download_button",
    "description": (
        "Navigate to a page, click a download button by its visible text, and intercept "
        "the file download triggered by the click using Playwright's download API. "
        "Use this when get_page_snapshot reports has_download_buttons=true. "
        "Returns a content_id, pass it to convert_to_markdown next."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url":         {"type": "string", "description": "The page URL containing the download button."},
            "button_text": {"type": "string", "description": "The visible text of the download button, e.g. 'Download' or 'Download PDF'."}
        },
        "required": ["url", "button_text"]
    }
}

convert_to_markdown_tool = {
    "type": "function",
    "name": "convert_to_markdown",
    "description": (
        "Convert Playwright-fetched content (HTML or PDF) to clean Markdown using MarkItDown. "
        "Pass the content_id returned by click_and_capture or click_download_button. "
        "Returns a markdown_id and a short preview, pass the markdown_id to save_content_to_file."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "content_id": {"type": "string", "description": "The content_id returned by click_and_capture or click_download_button."}
        },
        "required": ["content_id"]
    }
}

save_content_to_file_tool = {
    "type": "function",
    "name": "save_content_to_file",
    "description": (
        "Save the converted Markdown to a local .md file. "
        "Pass the markdown_id returned by convert_to_markdown and a descriptive filename."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "markdown_id": {"type": "string", "description": "The markdown_id returned by convert_to_markdown."},
            "filename":    {"type": "string", "description": "Filename for the saved file, e.g. 'european-climate-law.md'."},
            "directory":   {"type": "string", "description": "Directory to save into. Defaults to 'climate_policy_docs'."}
        },
        "required": ["markdown_id", "filename"]
    }
}

# Tool dispatcher 

async def run_tool(name: str, args: dict) -> str:
    if name == "get_page_snapshot":
        result = await playwright_get_page_snapshot(**args)
    elif name == "click_and_capture":
        result = await playwright_click_and_capture(**args)
    elif name == "click_download_button":
        result = await playwright_click_download_button(**args)
    elif name == "convert_to_markdown":
        result = convert_to_markdown(**args)
    elif name == "save_content_to_file":
        result = save_content_to_file(**args)
    else:
        result = {"error": f"Unknown tool: {name}"}
    return json.dumps(result)


# Project topic used by the agent to decide whether content is worth saving 
PROJECT_TOPIC = """
This is an EU Climate Policy Q&A RAG system. It helps students and policy learners
navigate official EU climate policy documents. Relevant content includes:
  - EU climate legislation and regulations (e.g. European Climate Law, EU ETS)
  - EU climate targets (2030, 2040, net-zero 2050)
  - EU Commission communications and proposals on climate policy
  - Staff working documents, factsheets, Q&As, and press releases about EU climate policy
  - The Clean Industrial Deal and related EU green transition documents

IRRELEVANT content includes: cookie policies, login pages, navigation-only pages,
general EU portal homepages, unrelated legislation, press releases about non-climate topics,
and any document that does not substantively discuss EU climate policy.
"""

FETCH_AGENT_INSTRUCTIONS = f"""
You are a document-fetching agent. Given a URL, your goal is to retrieve, convert, and
save the document as Markdown, but only if it is relevant to the project topic below.

PROJECT TOPIC:
{PROJECT_TOPIC}

Follow these steps in order:
1. Call get_page_snapshot on the starting URL to inspect the page.
   Use the page title, URL, and link texts to make a preliminary relevance judgment.
   If the page is clearly unrelated to EU climate policy (e.g. a login page, a cookie
   notice, an unrelated press release), stop immediately and explain why.
   NOTE: if the URL is already a direct download link (e.g. EU Commission
   /document/download/<uuid>), get_page_snapshot will return has_downloadable_documents=true
   with that URL as the only document_link; proceed directly to step 2a.

2. Decide how to get the document based on the snapshot:
   a) If has_downloadable_documents=true → pick the best link from 'document_links'
      and call click_and_capture with its href.
      'document_links' includes both regular <a href> links AND <a role="button"> anchors
      whose href contains 'document' and 'download' (EU Commission pattern).
   b) If has_download_buttons=true → call click_download_button using the button text
      from 'download_buttons'. Playwright will intercept the downloaded file.
   c) If neither → look in 'links' for the link most likely to lead to the document
      (prefer EUR-Lex /TXT/HTML/ links, then full-text HTML pages, then PDFs)
      and call click_and_capture.

3. After click_and_capture or click_download_button returns:
   - If format is "pdf" or "binary" → go directly to step 4.
   - If format is "html" → call get_page_snapshot on the returned URL to check
     whether the landed page itself has downloadable documents or download buttons.
     If yes, repeat step 2 on that URL.
     If no, the page is the document itself; proceed to step 4 with the content_id
     already in hand.

4. Call convert_to_markdown with the content_id.

5. RELEVANCE CHECK: read the title and preview returned by convert_to_markdown.
   Ask yourself: does this document substantively discuss EU climate policy?
   - YES → proceed to step 6.
   - NO  → do NOT call save_content_to_file. Discard the markdown_id and explain
           in one sentence why the content is not relevant.

6. Call save_content_to_file with the markdown_id and a descriptive .md filename
   derived from the document title shown in the preview.

7. Confirm what was saved and provide a one-sentence summary.

Stop as soon as the file has been saved or the relevance check fails.
"""


async def llm_fetch_document(url: str, max_turns: int = 12) -> str:
    """
    LLM-powered agent: Playwright fetches the content, MarkItDown converts it,
    result is saved to a .md file. Large content never passes through the LLM.
    """
    messages = [
        {"role": "system", "content": FETCH_AGENT_INSTRUCTIONS},
        {"role": "user",   "content": f"Fetch and save the document at: {url}"}
    ]
    tools = [
        get_page_snapshot_tool,
        click_and_capture_tool,
        click_download_button_tool,
        convert_to_markdown_tool,
        save_content_to_file_tool
    ]

    for turn in range(max_turns):
        print(f"\n [turn {turn+1}/{max_turns}] Asking LLM what to do next…")
        response = openai_client.responses.create(
            model="gpt-5.4-mini",
            input=messages,
            tools=tools
        )

        for item in response.output:
            if hasattr(item, "type") and item.type == "message":
                for block in getattr(item, "content", []):
                    text = getattr(block, "text", None)
                    if text:
                        print(f" LLM: {text.strip()}")

        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            print("Agent finished: no more tool calls.")
            return response.output_text

        for item in response.output:
            messages.append(item)

        for tc in tool_calls:
            args   = json.loads(tc.arguments)
            print(f" Calling tool: {tc.name}  args: { {k: str(v)[:80] for k, v in args.items()} }")
            result = await run_tool(tc.name, args)
            result_preview = json.loads(result)

            if tc.name == "get_page_snapshot":
                n_links  = len(result_preview.get("links", []))
                has_docs = result_preview.get("has_downloadable_documents", False)
                has_btns = result_preview.get("has_download_buttons", False)
                n_doc    = len(result_preview.get("document_links", []))
                n_btn    = len(result_preview.get("download_buttons", []))
                print(f"   → page: \"{result_preview.get('title', '')}\" - {n_links} links | "
                      f"doc links: {n_doc} (has_downloadable_documents={has_docs}) | "
                      f"download buttons: {n_btn} (has_download_buttons={has_btns})")
                for dl in result_preview.get("document_links", [])[:5]:
                    print(f"      link: {dl['text']!r}  →  {dl['href']}")
                for db in result_preview.get("download_buttons", [])[:5]:
                    print(f"      button: {db['text']!r}")
            elif tc.name in ("click_and_capture", "click_download_button"):
                print(f"   → fetched: \"{result_preview.get('title', '')}\" ({result_preview.get('format','?')}) - content_id: {result_preview.get('content_id','?')}")
            elif tc.name == "convert_to_markdown":
                print(f"   → converted: {result_preview.get('length', 0):,} chars - markdown_id: {result_preview.get('markdown_id','?')}")
                print(f"   preview: {result_preview.get('preview','')[:200].strip()!r}")
            elif tc.name == "save_content_to_file":
                print(f"   → saved to: {result_preview.get('path','?')}  ({result_preview.get('bytes',0):,} bytes)")
            else:
                print(f"   → {json.dumps(result_preview)[:140]}")

            messages.append({
                "type":    "function_call_output",
                "call_id": tc.call_id,
                "output":  result
            })

    return "Agent reached max turns without a final answer."


In [ ]:
test_link = list(enriched_sections.values())[0][13]
print("Testing LLM fetcher agent on:", test_link["url"])
fetched_content = await llm_fetch_document(test_link["url"])
print("\n Agent result:")
print(fetched_content)
